# Week 33, Day 4: Short-term plasticity as computation: memory traces and burst detection

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How long does the trace last?

Using `tm` from yesterday (redefined below), load a synapse with 20 spikes at 50 Hz and compute the efficacy of a probe spike at delays of 200, 500, 1000, 2000 and 4000 ms after the last loading spike, divided by the baseline U. Store it in `gain[delay]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def tm(times, U, tauD, tauF):
    u, x, last, out = U, 1.0, None, []
    for t in times:
        if last is not None:
            d = t - last
            x = 1 + (x - 1) * np.exp(-d / tauD)
            u = (U + (u - U) * np.exp(-d / tauF)) if tauF > 0 else U
        out.append(u * x)
        x = x - u * x
        if tauF > 0:
            u = u + U * (1 - u)
        last = t
    return np.array(out)

U, tauD, tauF = 0.2, 200.0, 1500.0
load = np.arange(20) * 20.0                    # 20 spikes at 50 Hz
gain = {}
for delay in (200, 500, 1000, 2000, 4000):
    gain[delay] = ...
    print(f"{delay:5d} ms after loading: efficacy {gain[delay]:.2f} times baseline")

In [ ]:
import numpy as np
def test_trace():
    "the loaded synapse is at least 2.5 times stronger than baseline 1 s after loading, and within 50% of baseline after 4 s"
    assert gain[1000] > 2.5 and gain[4000] < 1.5

def test_shape():
    "the trace first rises, as depression recovers, then falls as facilitation decays"
    assert gain[500] > gain[200] and gain[500] > gain[2000]

test_trace()
test_shape()
print("All checks passed")

### Exercise: Low-rate reactivation

For reactivation rates of 0.5, 1, 2, 3 and 5 Hz, drive the synapse with 60 regular spikes and store the steady efficacy, the last response divided by U, in `held[rate]`.

In [ ]:
held = {}
for rate in (0.5, 1, 2, 3, 5):
    held[rate] = ...
    print(f"reactivation at {rate} Hz: efficacy held at {held[rate]:.2f} times baseline")

In [ ]:
import numpy as np
def test_refresh():
    "reactivations at 1 Hz hold the efficacy above 1.5 times baseline, at 3 Hz above 2"
    assert held[1] > 1.5 and held[3] > 2

def test_cheap():
    "the steady efficacy at 3 Hz is within 15% of that at 5 Hz"
    assert abs(held[3] / held[5] - 1) < 0.15

test_refresh()
test_cheap()
print("All checks passed")

### Exercise: Who reads bursts?

Compare 21 isolated spikes at 2 Hz with 7 bursts of 3 spikes (5 ms apart, bursts every 1.5 s). For a facilitating synapse (U = 0.1, τ_rec = 50 ms, τ_F = 100 ms) and a depressing one (U = 0.5, τ_rec = 800 ms, no facilitation), compute the mean response per spike in bursts divided by that for isolated spikes, `ratio[name]`.

In [ ]:
isolated = np.arange(21) * 500.0
bursts = np.concatenate([b * 1500.0 + np.arange(3) * 5.0 for b in range(7)])
syn = {"facilitating": (0.1, 50.0, 100.0), "depressing": (0.5, 800.0, 0.0)}
ratio = {}
for name, p in syn.items():
    ratio[name] = ...
    print(f"{name:12s}: per spike, bursts give {ratio[name]:.2f} times what isolated spikes give")

In [ ]:
import numpy as np
def test_fac():
    "the facilitating synapse transmits more than 1.4 times as much per spike in bursts as in isolated spikes"
    assert ratio["facilitating"] > 1.4

def test_dep():
    "the depressing synapse transmits less per spike in bursts"
    assert ratio["depressing"] < 0.9

test_fac()
test_dep()
print("All checks passed")